# 🧪 Etapa 2 — Fine-Tuning com LoRA (4 Modelos)

Este notebook aplica **LoRA (Low-Rank Adaptation)** para ajustar 4 modelos de linguagem ao domínio do dataset gerado na Etapa 1.

### Modelos treinados:
| Modelo | Tipo | target_modules |
|--------|------|----------------|
| `EleutherAI/pythia-160m` | Causal (Decoder) | `query_key_value` |
| `EleutherAI/gpt-neo-125m` | Causal (Decoder) | `q_proj`, `v_proj` → `c_attn` |
| `google/flan-t5-base` | Seq2Seq (Enc-Dec) | `q`, `v` |
| `google/mt5-small` | Seq2Seq (Enc-Dec) | `q`, `v` |

### LoRA — Fundamentos:
$$\Delta W = B \cdot A, \quad B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times k}, r \ll \min(d,k)$$

$$h = Wx + \frac{\alpha}{r} BAx$$

Apenas as matrizes $A$ e $B$ são treinadas — o modelo base permanece **congelado**.


## 📦 1. Dependências


In [ ]:
!pip uninstall -y gradio
!pip install "pillow<12.0.0" "peft==0.10.0" "datasets==2.18.0" transformers accelerate torch -q


## 2. Importações


In [ ]:
import os
import json
import math
import torch
import matplotlib.pyplot as plt
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'PyTorch: {torch.__version__} | Dispositivo: {DEVICE}')


## 3. Carregamento e Preparação do Dataset

Carregamos o `dataset_gerado.jsonl` (gerado na Etapa 1 com o Gemma 2 2B).
Cada exemplo é convertido para o formato `Instruction: ... Output: ...` para modelos Causais,
e mantém os campos separados para modelos Seq2Seq.


In [ ]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto para modelos causais."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o dataset gerado
DATASET_FILE = "dataset_gerado.jsonl"
dataset = load_dataset('json', data_files=DATASET_FILE)
dataset = dataset.map(convert_to_hf_format)
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

print(f'Dataset carregado:')
print(f'  Treino:    {len(dataset["train"])} exemplos')
print(f'  Validação: {len(dataset["test"])} exemplos')
print(f'\nExemplo:')
print(dataset["train"][0]["text"])


## 4. Configuração dos 4 Modelos LoRA

### Justificativa dos Hiperparâmetros:
- **r=16**: rank moderado — equilíbrio entre capacidade de adaptação e eficiência computacional (seguindo o notebook base)
- **lora_alpha=32**: escalonamento α/r = 2.0, intensidade adequada para datasets pequenos
- **lora_dropout=0.05**: regularização leve nas matrizes LoRA (evita overfitting)
- **target_modules**: escolhidos conforme a arquitetura de cada modelo (detalhado abaixo)
  - Pythia: `query_key_value` (projeção combinada de Q, K, V)
  - GPT-Neo: usa `Conv1D`, mapeado via `c_attn` / `c_proj`
  - Flan-T5 / mT5: projeções `q` e `v` nas atenções do Encoder e Decoder


In [ ]:
MODELOS_CONFIG = [
    {
        "nome": "EleutherAI/pythia-160m",
        "tipo": "causal",
        "target_modules": ["query_key_value"],
        "save_model": "lora_causal_pythia_160m",
        "save_token": "tokenizer_pythia_160m",
        "descricao": "Pythia-160M — Causal LM, arquitetura GPT-NeoX. target: query_key_value"
    },
    {
        "nome": "EleutherAI/gpt-neo-125m",
        "tipo": "causal",
        "target_modules": ["q_proj", "v_proj"],
        "save_model": "lora_causal_gpt_neo_125m",
        "save_token": "tokenizer_gpt_neo_125m",
        "descricao": "GPT-Neo-125M — Causal LM, atenção Local+Global. target: q_proj, v_proj"
    },
    {
        "nome": "google/flan-t5-base",
        "tipo": "seq2seq",
        "target_modules": ["q", "v"],
        "save_model": "lora_seq2seq_flan_t5_base",
        "save_token": "tokenizer_flan_t5_base",
        "descricao": "Flan-T5-Base — Seq2Seq (Enc-Dec), fine-tuned em 1800+ tasks. target: q, v"
    },
    {
        "nome": "google/mt5-small",
        "tipo": "seq2seq",
        "target_modules": ["q", "v"],
        "save_model": "lora_seq2seq_mt5_small",
        "save_token": "tokenizer_mt5_small",
        "descricao": "mT5-Small — Seq2Seq multilíngue (101 idiomas), bom para texto em PT. target: q, v"
    }
]

print('Configuração dos modelos:')
for m in MODELOS_CONFIG:
    print(f'  [{m["tipo"].upper():<8}] {m["nome"]:<30} → {m["descricao"]}')


## 5. Inferência ANTES do Fine-Tuning (Baseline)

Para cada modelo, executamos uma instrução de teste para registrar o comportamento base **antes** do LoRA.


In [ ]:
INSTRUCAO_TESTE = dataset["train"][0]["text"].split("\n")[0].replace("Instruction: ", "")
print(f'Instrução de teste: {INSTRUCAO_TESTE}\n')

def generate_response_causal(model, tokenizer, instruction, max_new_tokens=60):
    prompt = f"Instruction: {instruction}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full.split("Output:")[-1].strip()

def generate_response_seq2seq(model, tokenizer, instruction, max_new_tokens=60):
    inputs = tokenizer(f"Instruction: {instruction}", return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)

respostas_antes = {}
for config in MODELOS_CONFIG:
    print(f'📋 {config["nome"]} (ANTES):')
    try:
        tok = AutoTokenizer.from_pretrained(config["nome"])
        if config["tipo"] == "causal" and tok.pad_token is None:
            tok.pad_token = tok.eos_token
        if config["tipo"] == "causal":
            m = AutoModelForCausalLM.from_pretrained(config["nome"])
            resp = generate_response_causal(m, tok, INSTRUCAO_TESTE)
        else:
            m = AutoModelForSeq2SeqLM.from_pretrained(config["nome"])
            resp = generate_response_seq2seq(m, tok, INSTRUCAO_TESTE)
        respostas_antes[config["nome"]] = resp
        print(f'  → {resp[:120]}')
        del m
    except Exception as e:
        print(f'  ERRO: {e}')
    print()


## 6. Fine-Tuning LoRA — Loop dos 4 Modelos

Cada modelo é treinado sequencialmente, com os adaptadores LoRA salvos em diretórios separados.

> ⚠️ O treinamento é longo. Recomenda-se GPU com ≥ 8GB VRAM. Em CPU espera-se várias horas.


In [ ]:
historico_loss = {}  # Armazena loss por época de cada modelo

for config in MODELOS_CONFIG:
    print(f"\n{'='*60}")
    print(f"🚀 TREINANDO: {config['nome']}")
    print(f"   Tipo: {config['tipo'].upper()} | target_modules: {config['target_modules']}")
    print(f"{'='*60}")

    model_name = config["nome"]
    tipo = config["tipo"]

    # --- Tokenizador ---
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tipo == "causal" and tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- Modelo base ---
    if tipo == "causal":
        base_model = AutoModelForCausalLM.from_pretrained(model_name)
    else:
        base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # --- Prepara para LoRA ---
    model = prepare_model_for_kbit_training(base_model)

    task_type = TaskType.CAUSAL_LM if tipo == "causal" else TaskType.SEQ_2_SEQ_LM

    lora_config = LoraConfig(
        r=16,                         # Rank: equilíbrio adaptação × eficiência
        lora_alpha=32,                # Alpha: escalonamento α/r = 2.0
        target_modules=config["target_modules"],
        lora_dropout=0.05,            # Regularização leve
        bias="none",
        task_type=task_type,
        inference_mode=False
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # --- Tokenização ---
    if tipo == "causal":
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        def tokenize_fn(examples):
            return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
        tok_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
    else:
        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)
        def tokenize_seq2seq_fn(examples):
            model_inputs = tokenizer(examples["Instruction"], max_length=128, truncation=True, padding="max_length")
            labels = tokenizer(text_target=examples["Output"], max_length=64, truncation=True, padding="max_length")
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs
        tok_dataset = dataset.map(tokenize_seq2seq_fn, batched=True, remove_columns=dataset["train"].column_names)

    # --- Argumentos de Treinamento ---
    output_dir = f"./results_{config['save_model']}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        learning_rate=1e-3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=10,
        weight_decay=0.01,
        logging_steps=5,
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        fp16=(torch.cuda.is_available() and config[tipo] == causal)
    )

    # --- Trainer ---
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_dataset["train"],
        eval_dataset=tok_dataset["test"],
        data_collator=data_collator,
    )

    trainer.train()

    # Armazena histórico de loss
    logs = [x for x in trainer.state.log_history if "loss" in x]
    historico_loss[config["nome"]] = [(x.get("epoch", i), x["loss"]) for i, x in enumerate(logs)]

    # --- Salvar ---
    model.save_pretrained(config['save_model'])
    tokenizer.save_pretrained(config['save_token'])
    print(f"✅ Salvo em '{config['save_model']}' e '{config['save_token']}'")

    del model, base_model, tokenizer, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n✅ Todos os modelos treinados!')


## 7. Gráficos de Loss por Época

Visualizamos a convergência do treinamento para os 4 modelos.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, (model_name, history) in enumerate(historico_loss.items()):
    if history:
        epochs = [h[0] for h in history]
        losses = [h[1] for h in history]
        axes[i].plot(epochs, losses, 'o-', color='steelblue', linewidth=2)
        axes[i].set_title(model_name.split('/')[-1], fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Época')
        axes[i].set_ylabel('Loss')
        axes[i].grid(True, alpha=0.3)
    else:
        axes[i].text(0.5, 0.5, 'Sem dados', ha='center', va='center', transform=axes[i].transAxes)

plt.suptitle('Training Loss por Época — 4 Modelos LoRA', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('loss_por_epoca.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Gráfico salvo: loss_por_epoca.png')


## 8. Inferência APÓS o Fine-Tuning

Comparamos as respostas do modelo base com o modelo fine-tunado, usando **a mesma instrução de teste**, para evidenciar o ganho de especialização no domínio.


In [ ]:
from peft import PeftModel

print(f'Instrução de teste: {INSTRUCAO_TESTE}\n')

for config in MODELOS_CONFIG:
    print(f"{'='*50}")
    print(f"🔍 {config['nome']}")

    try:
        tok = AutoTokenizer.from_pretrained(config["save_token"])
        if config["tipo"] == "causal" and tok.pad_token is None:
            tok.pad_token = tok.eos_token

        if config["tipo"] == "causal":
            base = AutoModelForCausalLM.from_pretrained(config["nome"])
            ft_model = PeftModel.from_pretrained(base, config["save_model"])
            resp_depois = generate_response_causal(ft_model, tok, INSTRUCAO_TESTE)
        else:
            base = AutoModelForSeq2SeqLM.from_pretrained(config["nome"])
            ft_model = PeftModel.from_pretrained(base, config["save_model"])
            resp_depois = generate_response_seq2seq(ft_model, tok, INSTRUCAO_TESTE)

        resp_antes = respostas_antes.get(config["nome"], "N/A")
        print(f'  ANTES : {resp_antes[:120]}')
        print(f'  DEPOIS: {resp_depois[:120]}')
        del base, ft_model
    except Exception as e:
        print(f'  ERRO: {e}')
    print()


## 6. Exportando para o Google Drive / Download (Colab)
Se você estiver rodando no Google Colab, rode a célula abaixo para compactar todos os 4 modelos treinados em um único arquivo ZIP. Depois, basta clicar com o botão direito no `modelos_treinados.zip` na barra lateral e fazer o Download para o seu computador!

In [ ]:
!zip -r modelos_treinados.zip lora_causal_* lora_seq2seq_* tokenizer_*